# 05 — Evaluation (CRISP-DM)

Avaliação honesta do campeão publicado pelo notebook 04. Três compromissos:

1. **Factual vs. contrafactual** — separar o que é medido (demanda observada vs.
   prevista) do que é estimado (lucro sob descontos que não ocorreram).
2. **Auditoria de vazamento re-verificada** — o guardrail é checado *aqui também*,
   não só no treino: nenhuma coluna vetada pode ser feature.
3. **Contexto de negócio** — comparar com baselines ingênuos e com a curva de
   desconto, para calibrar a expectativa do ganho.

Consumimos apenas artefatos publicados (`models/`), sem re-treinar nada.

## 1) Artefatos e proveniência

O snapshot de métricas e o cartão de modelo carregam a proveniência do run
(seed, hash dos dados, janelas temporais), garantindo reprodutibilidade.

In [1]:
import json

import pandas as pd

from src.config.paths import MODELS_DIR

snapshot = json.loads((MODELS_DIR / "metrics_snapshot.json").read_text(encoding="utf-8"))
cartao = json.loads((MODELS_DIR / "model_card.json").read_text(encoding="utf-8"))

prov = snapshot["proveniencia"]
print(f"Modelo campeão : {snapshot['modelo']}")
print(f"Seed           : {prov['seed']} | hash dos dados: {prov['hash_dados']}")
print("Linhas por split:", prov["linhas_por_split"])
for nome, (inicio, fim) in prov["periodos"].items():
    print(f"  {nome:>9}: {inicio} → {fim}")

Modelo campeão : RidgeCV
Seed           : 42 | hash dos dados: f495cd37a2149bc6
Linhas por split: {'treino': 26256, 'validacao': 4416, 'teste': 4416}
     treino: 2024-01-01 → 2025-06-30
  validacao: 2025-07-01 → 2025-09-30
      teste: 2025-10-01 → 2025-12-31


## 2) Auditoria de vazamento re-verificada

O contrato de features é validado *fail-closed* também na avaliação. Se qualquer
coluna vetada por vazamento (`quantidade_vendida_mes`, `estoque_disponivel`,
`custo_marketing_unitario`, `ruptura_flag`) ou o próprio alvo aparecer como
feature, a célula aborta.

In [2]:
from src.modeling import train_pipeline as tp

base = tp.carregar_base_preparada()
tp.validar_contrato_features(base)  # fail-closed: vazamento aborta o notebook

vetadas = set(tp.colunas_com_vazamento())
features = set(cartao["features_numericas"]) | set(cartao["features_categoricas"])
assert not (features & vetadas), f"Vazamento no contrato: {sorted(features & vetadas)}"
assert cartao["alvo"] not in features, "O alvo não pode ser feature."
assert features <= set(base.columns), "Features do cartão ausentes na base."

print(f"Guardrail OK — {len(features)} features leakage-safe contra {len(vetadas)} colunas vetadas.")
print("Colunas vetadas:", sorted(vetadas))

Guardrail OK — 24 features leakage-safe contra 4 colunas vetadas.
Colunas vetadas: ['custo_marketing_unitario', 'estoque_disponivel', 'quantidade_vendida_mes', 'ruptura_flag']


## 3) Métricas factuais vs. baselines ingênuos

O RMSE do modelo é comparado com duas referências sem modelo: prever a média
global do treino e prever a média por (loja, produto). É o teste de honestidade
mínimo — mostrar que o modelo agrega valor além do trivial.

In [3]:
teste = snapshot["teste"]
baselines = snapshot["baselines"]

comparativo = pd.DataFrame(
    [
        {"referência": f"modelo ({snapshot['modelo']})", "rmse": teste["rmse"]},
        {"referência": "baseline média global", "rmse": baselines["rmse_media_global"]},
        {"referência": "baseline média por SKU", "rmse": baselines["rmse_media_sku"]},
    ]
)
comparativo["ganho_vs_modelo_%"] = (comparativo["rmse"] / teste["rmse"] - 1.0) * 100.0
comparativo.round(4)

,referência,rmse,ganho_vs_modelo_%
0,modelo (RidgeCV),0.8439,0.0000
1,baseline média global,2.6314,211.8103
2,baseline média por SKU,1.4515,71.9938


In [4]:
print(f"R² (teste) : {teste['r2']:.3f}")
print(f"MAE (teste): {teste['mae']:.3f}  (≈ {teste['mae_ratio']:.1%} da demanda média)")
print(f"RMSE (teste): {teste['rmse']:.3f} vs. média por SKU {baselines['rmse_media_sku']:.3f}")

R² (teste) : 0.897
MAE (teste): 0.490  (≈ 26.2% da demanda média)
RMSE (teste): 0.844 vs. média por SKU 1.451


## 4) Honestidade do contrafactual

O ganho de lucro da política é uma **estimativa** (a demanda sob descontos que
não ocorreram é predita pelo modelo). Reportamos validação e teste lado a lado,
com a nota de honestidade gravada no snapshot.

In [5]:
colunas = ["lucro_baseline", "lucro_politica", "uplift_pct", "desconto_medio", "pct_linhas_com_desconto"]
negocio = pd.DataFrame(
    {
        "validacao": {k: snapshot["validacao"][k] for k in colunas},
        "teste": {k: teste[k] for k in colunas},
    }
)
print(snapshot["honestidade"])
negocio.round(4)

Métricas de demanda no teste são factuais (demanda observada vs. prevista). O lucro da política é uma estimativa contrafactual (a demanda sob descontos alternativos é prevista pelo modelo), não um resultado realizado; use o baseline ingênuo como calibração do ganho esperado.


,validacao,teste
lucro_baseline,143.1946,156.4299
lucro_politica,149.5341,162.1246
uplift_pct,4.4272,3.6404
desconto_medio,0.0547,0.0511
pct_linhas_com_desconto,0.5562,0.5288


## 5) Curva de negócio (demanda/receita/lucro por desconto)

A curva **média agregada** é monotonicamente decrescente: no agregado, o preço
cheio já é o melhor. O uplift da política vem da **heterogeneidade por linha** —
em SKUs/dias específicos, um desconto pequeno compra volume suficiente para
superar a queda de margem. É exatamente essa diferença que o otimizador captura.

In [6]:
curva = pd.read_csv(MODELS_DIR / "business_curve.csv")
curva.round(4)

,desconto_pct,qtd_media_prevista,receita_media_prevista,lucro_medio_previsto
0,0.00,1.7566,249.1215,156.4299
1,0.01,1.7844,248.9529,155.3869
2,0.02,1.8123,248.7374,154.2971
3,0.03,1.8402,248.4751,153.1603
4,0.04,1.8680,248.1658,151.9766
5,0.05,1.8959,247.8096,150.7460
6,0.06,1.9238,247.4067,149.4686
7,0.07,1.9517,246.9566,148.1442
8,0.08,1.9795,246.4597,146.7728
9,0.09,2.0074,245.9158,145.3545


In [7]:
monotonica = curva["lucro_medio_previsto"].is_monotonic_decreasing
otimo_medio = curva.loc[curva["lucro_medio_previsto"].idxmax()]
print(f"Curva média monotônica decrescente: {monotonica}")
print(f"Ótimo médio agregado: desconto={otimo_medio['desconto_pct']:.2f}")
print(f"Demanda média: {curva['qtd_media_prevista'].iloc[0]:.2f} (0%) → {curva['qtd_media_prevista'].iloc[-1]:.2f} (12%)")

Curva média monotônica decrescente: True
Ótimo médio agregado: desconto=0.00
Demanda média: 1.76 (0%) → 2.09 (12%)


## 6) Importância das features

Importâncias legíveis por feature original (bloco one-hot somado de volta),
para leitura de negócio. `promocao_tipo` domina porque é a alavanca de volume
mais forte da base; as variáveis de preço vêm em seguida, coerentes com o
framing de elasticidade.

In [8]:
importancias = pd.Series(cartao["importancia_features"], name="importancia").sort_values(ascending=False)
importancias.head(12).round(4)

promocao_tipo        23.6229
produto               6.3555
tamanho_embalagem     3.7000
preco_original        3.5972
tipo_produto          3.4368
promocao_flag         3.3142
preco_final           3.1905
categoria             1.5141
evento_especial       0.6093
desconto_pct          0.3699
loja                  0.2632
perfil_loja           0.2632
Name: importancia, dtype: float64

## 7) Casos por linha no teste

O detalhamento por linha do Q4/2025 mostra *onde* a política age: quantas linhas
recebem desconto e o lucro médio com e sem a política.

In [9]:
cenarios = pd.read_parquet(MODELS_DIR / "test_predictions.parquet")
print(f"Casos no teste: {len(cenarios):,}")
cenarios.head(8).round(3)

Casos no teste: 4,416


,data,loja,produto,preco_original,preco_minimo,custo_producao,quantidade_vendida_dia,demanda_prevista,desconto_otimo,preco_otimo,lucro_politica,lucro_baseline
0,2025-10-01,Unidade A,Docinho de Festa (cento),240.0,93.56,71.75,1,1.295,0.12,211.20,229.136,217.820
1,2025-10-01,Unidade A,"Empadao Congelado 1,2 kg",98.0,52.78,38.67,1,1.178,0.12,86.24,91.345,69.919
2,2025-10-01,Unidade A,Fatia de Torta,24.0,11.20,8.08,6,8.368,0.00,24.00,133.226,133.226
3,2025-10-01,Unidade A,Kit Docura 3,389.0,160.97,126.45,0,0.532,0.00,389.00,139.584,139.584
4,2025-10-01,Unidade A,Kit Festa 2,379.0,195.39,141.79,1,0.934,0.00,379.00,221.597,221.597
5,2025-10-01,Unidade A,Mini Cake Individual,49.9,21.10,16.02,7,5.565,0.00,49.90,188.551,188.551
6,2025-10-01,Unidade A,Quiche Base 500 g,38.0,18.68,14.81,2,2.102,0.12,33.44,55.990,48.734
7,2025-10-01,Unidade A,Salgadinho Assado (cento),189.0,102.58,75.59,1,1.389,0.12,166.32,148.751,130.143


In [10]:
cenarios["recebeu_desconto"] = cenarios["desconto_otimo"] > 0
resumo = cenarios.groupby("recebeu_desconto")[["lucro_politica", "lucro_baseline"]].mean().round(3)
print(f"Linhas com desconto: {cenarios['recebeu_desconto'].mean():.1%}")
resumo

Linhas com desconto: 52.9%


,lucro_politica,lucro_baseline
recebeu_desconto,,
False,164.908,164.908
True,159.644,148.874


## Conclusão

- **Factual**: o modelo reduz o RMSE de demanda bem abaixo dos baselines
  ingênuos (média global e média por SKU) no teste Q4 — valor preditivo real.
- **Contrafactual**: o uplift de lucro da política é uma **estimativa**,
  reportada com nota de honestidade e calibrada contra o preço de tabela.
- **Vazamento**: guardrail de features re-verificado na avaliação; nenhuma
  coluna vetada entra no modelo.

A Fase 4 consome estes mesmos artefatos (`business_curve.csv`,
`metrics_snapshot.json`, `test_predictions.parquet`, `best_model.pkl`) para o app
e o site, sem re-treinar.